# In-Class Activity: Build Your Own MCP Server 🔌

**Objective:** Build your own MCP server that turns an API into tools Gemini can use.

You will:
1. Choose an API.
2. Build a STDIO MCP server with at least two tools.
3. Handle errors, timeouts, empty results, and rate limits.
4. Test the server with an MCP client and with Gemini.

Let's get started! 🚀

## Part 0: What is MCP?

MCP (Model Context Protocol) is a standard that lets AI applications connect to external tools and data, such as APIs, databases, and files. Think of it as USB-C for AI.

An MCP server can provide:
1. **Resources:** data clients can read, such as files or API responses.
2. **Tools:** functions an LLM can call to take actions.
3. **Prompts:** templates that help users with specific tasks.

Today you will build a server that provides tools.

### Example Workflow: Asking Gemini a Question
1. The client sends your question to Gemini.
2. Gemini decides whether it needs one of the available tools.
3. The client calls that tool through your MCP server.
4. The server runs the tool and returns the result.
5. Gemini uses the result to write the final answer.

The client runs your server as a separate program, and they communicate through `stdin` and `stdout`. This is called the **STDIO transport**.

### References

- [What is MCP?](https://modelcontextprotocol.io/docs/getting-started/intro)
- [Build an MCP server](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server)
- [Python SDK](https://github.com/modelcontextprotocol/python-sdk)

## Part 1: Setup and Configuration

In [ ]:
%pip install -q "mcp>=2.2,<3" "google-genai>=1.50"

Add your Gemini API key to Colab **Secrets** (🔑 in the left sidebar) as `GEMINI_API_KEY`. Get a key [here](https://ai.google.dev/gemini-api/docs/api-key) if you don't have one.

In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GEMINI_API_KEY')
print(f"Using API Key: {API_KEY[:4]}...{API_KEY[-4:]}")  # Print only the first and last 4 characters for security

In [ ]:
from importlib.metadata import version

print("mcp:", version("mcp"))
print("google-genai:", version("google-genai"))
print("✅ Setup Complete!")

### ⚠️ A Note on Rate Limits

We're using `gemini-3.5-flash-lite` because it's a free-tier model, but rate limits still apply (and vary by model).

See the [Rate Limits docs](https://ai.google.dev/gemini-api/docs/rate-limits) for details. Three things to watch:

- **RPM (Requests Per Minute):** API calls allowed per 60-second window.
- **TPM (Tokens Per Minute):** input tokens allowed per 60-second window.
- **RPD (Requests Per Day):** total requests allowed per day (resets at midnight Pacific time).

Exceeding any one triggers a `429 RESOURCE_EXHAUSTED` error. If that happens, wait a few seconds and try again, and don't send too many requests in a short time.

Your API has rate limits too. You'll handle them in Part 3.

---

## Part 2: Choose an API

Pick an API. These need no sign-up:

| Category | API | Base URL | Example endpoints |
|---|---|---|---|
| Weather | [Open-Meteo](https://open-meteo.com/en/docs) | `https://api.open-meteo.com` | `/v1/forecast?latitude=..&longitude=..` |
| Movies/TV | [TVmaze](https://www.tvmaze.com/api) | `https://api.tvmaze.com` | `/search/shows?q=..`, `/shows/{id}/episodes` |
| Books | [Open Library](https://openlibrary.org/developers/api) | `https://openlibrary.org` | `/search.json?q=..`, `/search/authors.json?q=..` |
| GitHub issues | [GitHub REST](https://docs.github.com/en/rest) | `https://api.github.com` | `/repos/{owner}/{repo}/issues`, `/search/issues?q=..` |
| Currency | [Frankfurter](https://frankfurter.dev) | `https://api.frankfurter.dev` | `/v1/latest?base=USD`, `/v1/{date}?base=USD` |
| Crypto | [CoinGecko](https://docs.coingecko.com) | `https://api.coingecko.com` | `/api/v3/simple/price?ids=..&vs_currencies=usd`, `/api/v3/search?query=..` |
| Holidays | [Nager.Date](https://date.nager.at/Api) | `https://date.nager.at` | `/api/v3/PublicHolidays/{year}/{country}`, `/api/v3/NextPublicHolidays/{country}` |
| Sports | [TheSportsDB](https://www.thesportsdb.com/free_sports_api) | `https://www.thesportsdb.com` | `/api/v1/json/123/searchteams.php?t=..`, `/api/v1/json/123/eventslast.php?id=..` |
| Wikipedia | [Wikipedia REST](https://api.wikimedia.org/wiki/Core_REST_API) | `https://en.wikipedia.org` | `/w/rest.php/v1/search/page?q=..`, `/api/rest_v1/page/summary/{title}` |

You can also use an API that needs a key. Part 4 shows how to pass it in.

### Your API plan

Double-click to fill in:

| | |
|---|---|
| API | TODO |
| Base URL | TODO |
| Endpoint 1 | TODO: path, parameters, what it returns |
| Endpoint 2 | TODO: path, parameters, what it returns |
| Rate limit | TODO |

Call each endpoint with `httpx2`, which comes with the MCP SDK. Also try one input that returns nothing and one that returns an error.

In [ ]:
import json
import httpx2

# TODO: Call your first endpoint and print the JSON.
# Hint: httpx2.get(url, params={...}, timeout=10).json()

# TODO: Call your second endpoint and print the JSON.

# TODO: Try an input that returns an empty result.

# TODO: Try an input that returns an error. Print the status code.

---

## Part 3: Build Your Server

`my_server.py` has four parts:
1. **Setup:** imports and `MCPServer("name")`.
2. **Helper:** `make_request` calls the API and turns failures into a `ToolError`, a readable error for the model.
3. **Tools:** each `@mcp.tool()` function is a tool. Type hints become its input schema, and the docstring becomes its description.
4. **Run:** `mcp.run(transport="stdio")`.

The base URL, API key, and timeout come from environment variables.

### ⚠️ Never use print() in a STDIO server
stdout carries the MCP messages, so `print()` breaks the connection. Use `logging`, which writes to stderr.

### Your tasks
Fill in the TODOs: the base URL, the error handling, and at least two tools. Rerun the cell after each change.

In [ ]:
%%writefile my_server.py
"""TODO: One line describing your server and the API it uses.

Endpoints used:
  - TODO: GET /...   what it does
  - TODO: GET /...   what it does
"""

import logging
import os
from typing import Any

import httpx2
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

# logging writes to stderr, which is safe for STDIO servers. Never use print().
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize MCPServer
mcp = MCPServer("my-server")  # TODO: give your server a short name

# Settings come from environment variables, with defaults.
API_BASE_URL = os.environ.get("API_BASE_URL", "https://...")  # TODO: your API's base URL
API_KEY = os.environ.get("API_KEY")  # only needed if your API requires a key
HTTP_TIMEOUT = float(os.environ.get("HTTP_TIMEOUT", "10"))
USER_AGENT = "swe270p-mcp-server/1.0"


async def make_request(path: str, params: dict[str, Any] | None = None) -> Any:
    """Make a request to the API with error handling."""
    headers = {"User-Agent": USER_AGENT}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"  # change this if your API expects the key elsewhere
    url = f"{API_BASE_URL}{path}"

    async with httpx2.AsyncClient() as client:
        try:
            response = await client.get(url, params=params, headers=headers, timeout=HTTP_TIMEOUT)
        except httpx2.TimeoutException:
            # TODO: log a warning and raise ToolError with a friendly message.
            raise
        except httpx2.RequestError:
            # TODO: log an error and raise ToolError with a friendly message.
            raise

    # TODO: If the status code is 429, raise ToolError telling the user to wait (rate limit).
    # TODO: If the status code is 404 or another 4xx/5xx, raise ToolError with a clear message.
    return response.json()


# TODO: Tool 1. Use @mcp.tool(), type hints, and a docstring with an Args section.
#   Validate the inputs, call make_request(...), handle an empty result, and return a short string.


# TODO: Tool 2.


if __name__ == "__main__":
    mcp.run(transport="stdio")

Check that the file imports without errors:

In [ ]:
import sys

!{sys.executable} -c "import my_server; print('✅ my_server.py imports cleanly')"

---

## Part 4: Test with an MCP Client

Start your server, connect to it, and list its tools. Server logs go to `server.log`.

In [ ]:
import os
from mcp import Client
from mcp.client.stdio import StdioServerParameters, stdio_client

# Extra environment variables for the server, e.g. {"API_KEY": userdata.get("MY_API_KEY")}
SERVER_ENV = {}


def connect_to_server(env: dict[str, str] | None = None) -> Client:
    """Start my_server.py as a subprocess and return an MCP client connected over STDIO.

    Args:
        env: Extra environment variables for this run (used in Part 5 to simulate failures).
    """
    params = StdioServerParameters(command=sys.executable, args=["my_server.py"], env={**SERVER_ENV, **(env or {})})
    # Colab's sys.stderr isn't a real file, so send the server's stderr (its logs) to a file.
    return Client(stdio_client(params, errlog=open("server.log", "a")))


def result_text(result) -> str:
    """Join the text blocks of a CallToolResult."""
    return "\n".join(block.text for block in result.content if block.type == "text")


async with connect_to_server() as client:
    print("Connected to:", client.server_info.name)
    for tool in (await client.list_tools()).tools:
        print(f"\n🔧 {tool.name}")
        print("   description:", tool.description.strip().splitlines()[0])
        print("   input schema:", json.dumps(tool.input_schema["properties"]))

This is what the model sees. Now call your tools yourself:

In [ ]:
async with connect_to_server() as client:
    # TODO: Call each of your tools with a normal input and print the result.
    # Example: result = await client.call_tool("tool_name", {"arg": "value"})
    #          print(result.is_error, result_text(result))
    pass

Check the server logs:

In [ ]:
!tail -n 20 server.log

---

## Part 5: Test Failure Cases

Each check should return a readable message, not crash or hang. These environment variables simulate failures:
- `HTTP_TIMEOUT=0.001`: timeout
- `API_BASE_URL=https://no-such-host.invalid`: unreachable API
- `API_BASE_URL=http://127.0.0.1:8765`: a fake API that always returns 429

In [ ]:
# A fake API that always returns 429 Too Many Requests.
import subprocess, textwrap

fake_api_code = textwrap.dedent('''
    from http.server import BaseHTTPRequestHandler, HTTPServer
    class AlwaysRateLimited(BaseHTTPRequestHandler):
        def do_GET(self):
            self.send_response(429)
            self.end_headers()
            self.wfile.write(b'{"error": "Too many requests"}')
        def log_message(self, *args):
            pass
    HTTPServer(("127.0.0.1", 8765), AlwaysRateLimited).serve_forever()
''')
with open("fake_rate_limited_api.py", "w") as f:
    f.write(fake_api_code)
fake_api = subprocess.Popen([sys.executable, "fake_rate_limited_api.py"])
print("Fake 429 API running on http://127.0.0.1:8765")

In [ ]:
# TODO: Fill in a tool name and arguments for each check.
checks = [
    ("Empty result",       None,                                             "TOOL_NAME", {}),
    ("Invalid input",      None,                                             "TOOL_NAME", {}),
    ("Timeout",            {"HTTP_TIMEOUT": "0.001"},                        "TOOL_NAME", {}),
    ("API unreachable",    {"API_BASE_URL": "https://no-such-host.invalid"}, "TOOL_NAME", {}),
    ("Rate limited (429)", {"API_BASE_URL": "http://127.0.0.1:8765"},        "TOOL_NAME", {}),
]

In [ ]:
for label, env, tool, args in checks:
    async with connect_to_server(env=env) as client:
        result = await client.call_tool(tool, args)
    print(f"[{label}] is_error={result.is_error}")
    print("   ", result_text(result).replace("\n", "\n    "))

### 🤔 Questions (TODO)
1. Which cases should return `is_error=True`? Is "nothing found" an error?
2. What rejects a string passed where your tool expects a number?

### Your Answers (Write your answer for each question)
  1.
  2.

In [ ]:
fake_api.terminate()  # stop the fake 429 API

---

## Part 6: Let Gemini Use Your Tools

Now Gemini picks the tools. The code:
1. Gets the tool list with `list_tools`.
2. Converts each tool to a Gemini `FunctionDeclaration`.
3. Sends the question and tools to Gemini.
4. Runs the tools Gemini asks for with `call_tool` and sends back the results.
5. Repeats until Gemini answers in text.

References: [Gemini function calling](https://ai.google.dev/gemini-api/docs/function-calling) · [google-genai SDK](https://googleapis.github.io/python-genai/)

In [ ]:
from google import genai
from google.genai import types

gemini = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash-lite"

SYSTEM_INSTRUCTION = (
    "You are a helpful assistant. Always use the provided tools to get real data; "
    "never make it up. If a tool returns an error, explain it to the user in plain words."
)


def mcp_tools_to_gemini(mcp_tools) -> types.Tool:
    """Convert a list of MCP tools (from client.list_tools().tools) into a Gemini Tool."""
    return types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name=tool.name,
                description=tool.description or "",
                parameters_json_schema=tool.input_schema,  # MCP already gives us a JSON Schema
            )
            for tool in mcp_tools
        ]
    )


print("✅ Gemini client ready")

In [ ]:
async def run_agent(question: str, client: Client, max_steps: int = 6) -> str:
    """Answer a question with Gemini, letting it call MCP tools as needed.

    Args:
        question: The user's question.
        client: A connected MCP client.
        max_steps: Maximum number of model calls, to stop endless tool loops.

    Returns:
        Gemini's final text answer.

    Hints:
        - config = types.GenerateContentConfig(tools=[mcp_tools_to_gemini(...)], system_instruction=SYSTEM_INSTRUCTION)
        - response = await gemini.aio.models.generate_content(model=MODEL, contents=contents, config=config)
        - response.function_calls is a list of calls (each has .name and .args), or empty/None when the model is done.
        - Add the model's turn to `contents` with response.candidates[0].content before adding tool results.
        - Print each tool call so you can see the invocation flow.
        - Send results back as types.Content(role="user", parts=[types.Part.from_function_response(name=..., response={...})]).
    """
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]
    pass  # TODO: Implement the agent loop

### Test Your Agent

Write two questions. At least one should need both tools. Watch which tools Gemini calls.

In [ ]:
# TODO: Write two questions about your API.
questions = [
    "TODO",
    "TODO",
]

async with connect_to_server() as client:
    for q in questions:
        print(f"\n🧑 {q}")
        print(f"🤖 {await run_agent(q, client)}")

### 🤔 Questions (TODO)
1. Ask something your tools can't answer. Does Gemini make something up?
2. Change a docstring to `"""Gets data."""`, rewrite the server, and ask again. What changes?
3. Report the cumulative Gemini API token usage across all API calls made during the assignment, including input, output, and reasoning tokens (if applicable). Use the usage_metadata field to retrieve this information.

### Your Answers (Write your answer for each question)
1.
2.
3.